# Week 3 — Watching Attention Work

**LLMs & You · Hampden-Sydney College · Fall 2026**

On Tuesday we read the paper *Attention is All You Need*. Today you will use the transformer network it describes and look
at the attention weights themselves — on sentences you choose.

The model below is a transformer doing **English → German translation**, which is
the task *Attention Is All You Need* was written about. It has six encoder layers,
six decoder layers, eight heads and a model dimension of 512 — the same numbers as
the base model in the paper's Table 3. You are not looking at an analogy of the
thing you read about. You are looking at the thing.

---

## This is not a worksheet

There are no steps to complete and nothing to hand in on Wednesday night. The code
below works. Running it top to bottom takes about ten minutes and teaches you very
little.

**What you are doing is looking for something to show on Thursday.** Every section
ends with a **Your turn** box: a question the code can answer but I have not
answered for you. Pick the ones you find interesting, change the sentences, and
find a result that surprised you.

Ten minutes each, in class. A confusion you can state precisely counts.

---

---
## 0. Setup

One cell, once per session. Colab forgets everything when the tab is closed, so if
you come back tomorrow, run this again. It downloads a 300 MB model the first time,
which takes a minute or two. You do not need an account, or API key. There is nothing to pay for.

In [ ]:
# Colab has the torch, numpy, pandas and matplotlib libraries already installed.
# These are the libraries it does not.
%pip install --quiet "transformers>=4.40" sentencepiece bertviz

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "Helsinki-NLP/opus-mt-en-de"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# attn_implementation="eager" is not optional. The faster default ("sdpa") never
# builds the attention matrix we want to look at, and silently hands back an empty
# tuple instead of an error. Every cell below would fail with no explanation.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, attn_implementation="eager")
model.eval()

print("ready:", MODEL_NAME)

---
## 1. Translate something

Before any attention, watch it do the job. `generate()` is the model writing German
one token at a time, each token conditioned on the English and on everything it has
written so far.

The second half of the cell prints the model's own dimensions. Compare them with the
paper.

In [ ]:
def translate(sentence, num_beams=4):
    """English in, German out."""
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**enc, num_beams=num_beams)
    return tokenizer.decode(out[0], skip_special_tokens=True)


for s in [
    "The trophy would not fit in the suitcase because it was too big.",
    "I know that he bought a new car yesterday.",
    "Attention is all you need.",
]:
    print(s)
    print("  ->", translate(s), "\n")

cfg = model.config
print("encoder layers :", cfg.encoder_layers)
print("decoder layers :", cfg.decoder_layers)
print("attention heads:", cfg.encoder_attention_heads)
print("d_model        :", cfg.d_model)
print("parameters     : %.1fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

!!! tip "Your turn"

    **Edit the cell above.** Click into it, and change the three sentences in the
    list — the strings between the square brackets. Then run it again: **Shift and
    Enter together**, or the play button that appears at the left of the cell.

    Nothing here is read-only, and nothing you can type will break it. If a cell
    stops working, run the setup cell at the top again and carry on.

    Every section below this one asks you to change a value and re-run. This is
    that skill, on the easiest possible thing to change, so get it working now.

    Translate something you can check — a sentence in a language you know, a
    sentence about Hampden-Sydney, a sentence with a name in it. Find one it gets
    wrong.

    The paper's base model has 65M parameters and this one has 74M. GPT-4-class
    models are roughly ten thousand times larger. Does the translation quality here
    surprise you in either direction?

---
## 2. The three kinds of attention

Figure 1 of the paper has three attention blocks, and the model returns all three:

| | what attends | to what |
|:---|:---|:---|
| **encoder self-attention** | each English token | every English token |
| **decoder self-attention** | each German token | German tokens *already written* |
| **cross-attention** | each German token | every English token |

One matrix per layer per head. To see attention over what the model *actually said*,
we translate first, then feed its own output back in — otherwise you are looking at
attention for a sentence it never produced.

In [ ]:
def look(sentence, num_beams=4):
    """Translate, then re-run the model on its own output to capture attention."""
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        generated = model.generate(**enc, num_beams=num_beams)
        out = model(
            input_ids=enc.input_ids,
            decoder_input_ids=generated,
            output_attentions=True,
        )
    english = tokenizer.convert_ids_to_tokens(enc.input_ids[0])
    german = tokenizer.convert_ids_to_tokens(generated[0])
    return english, german, out


def clean(tokens):
    """Marian marks a word start with the character U+2581. Drop it for display."""
    return [t.replace("\u2581", "") or "_" for t in tokens]


SENTENCE = "I know that he bought a new car yesterday."
english, german, out = look(SENTENCE)

print("English tokens:", clean(english))
print("German tokens :", clean(german))
print()
print("encoder self-attention:", len(out.encoder_attentions), "layers,",
      "each", tuple(out.encoder_attentions[0].shape), "= (batch, heads, from, to)")
print("decoder self-attention:", len(out.decoder_attentions), "layers, each",
      tuple(out.decoder_attentions[0].shape))
print("cross-attention       :", len(out.cross_attentions), "layers, each",
      tuple(out.cross_attentions[0].shape))

### The mask is a real thing

The paper says the decoder is masked so a position cannot attend to positions after
it — that is what stops the model reading the answer while writing it. You can check
the claim directly. Every number above the diagonal should be exactly zero.

In [ ]:
def heatmap(matrix, rows, cols, title, figsize=(7, 5.5)):
    """One attention matrix, drawn."""
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(matrix, cmap="viridis", vmin=0, aspect="auto")
    ax.set_xticks(range(len(cols)), clean(cols), rotation=90)
    ax.set_yticks(range(len(rows)), clean(rows))
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()


dec = out.decoder_attentions[0][0, 0].numpy()  # layer 0, head 0

above_diagonal = np.triu(dec, k=1)
print("largest weight above the diagonal:", above_diagonal.max())
print("each row sums to             :", np.round(dec.sum(axis=1)[:5], 6), "...")

heatmap(dec, german, german, "Decoder self-attention — layer 0, head 0")

!!! tip "Your turn"

    The triangle is the mask. Cover it with your hand and ask what the model would
    be able to do if it were not there — why would training break?

    Look along the first column. A lot of heads point at the very first token, which
    carries no meaning at all. What could a head be using it *for*?

---
## 3. Predict before you look

Here is the exercise that matters. Take a sentence where a pronoun is genuinely
ambiguous:

> *The trophy would not fit in the suitcase because it was too big.*

**Write down, now, which word you think `it` will attend to.** Then run the cell.

Averaging over the heads of a layer is a blunt instrument — it is the first thing
anyone does and it hides as much as it shows. Section 5 takes the average apart.

In [ ]:
AMBIGUOUS = "The trophy would not fit in the suitcase because it was too big."
english, german, out = look(AMBIGUOUS)

LAYER = 3
enc_att = out.encoder_attentions[LAYER][0].mean(axis=0).numpy()  # average the 8 heads

heatmap(enc_att, english, english,
        f"Encoder self-attention — layer {LAYER}, averaged over 8 heads")

WORD = "it"  # change this when you change the sentence
targets = [i for i, t in enumerate(english) if t.replace("\u2581", "") == WORD]
if not targets:
    raise SystemExit(f"{WORD!r} is not a token here. Tokens are: {clean(english)}")
row = targets[0]
weights = pd.Series(enc_att[row], index=clean(english)).sort_values(ascending=False)
print("What 'it' attends to, most to least:")
print(weights.head(6).round(3))

### Three things in that picture look wrong. None of them is.

**`</s>` takes the biggest weight, and it means nothing.**

`</s>` is the end-of-sentence token. The tokenizer appends it to every input so the
model knows where the sentence stops, and it carries no meaning of its own. In most
layers it takes the largest single weight in the `it` row. That looks like a bug and
is not.

Think back to Tuesday: softmax forces every row to add to exactly 1. A head with
nothing to contribute at this position still has to put its weight *somewhere* — it
has no way to abstain. So it learns to park the weight on a position that carries no
meaning, where adding it back changes the vector least. `</s>` is the most convenient
parking space in the sentence; the first token is the other one. Heads doing this are
sometimes called **no-ops**, and the habit is common enough to have a name: an
**attention sink**.

Reading these pictures is therefore a skill with one trick at the front of it:
discount the sinks first. What is left over is the part that means something.

**One token displays as `_`.**

Marian splits words it does not have whole. `trophy` is not in its vocabulary, so it
arrives as a word-start marker followed by `trophy`, and that marker cleans up to an
empty string which this notebook prints as `_`. It is half of `trophy`, not a word.

This is the Week 2 point about tokens not being words, turning up somewhere it costs
you something: the row you want to read may be split across two rows.

**No layer points `it` at the noun.**

Try all six. What you will not find is a layer where `it` puts its weight on `trophy`
or `suitcase` — the thing you did, without effort, before you ran the cell.

Sit with that, because the obvious conclusion is the wrong one. It does **not** mean
the model failed to resolve the pronoun. Translate the sentence and it commits to a
German pronoun carrying a gender, which is precisely a claim about which noun `it`
means. Something in there decided. Three things are going on:

- **You are looking at layer 3 of six, not at the model.** By the time a vector
  reaches layer 3 it is not the word `it` any more — it has been rewritten twice
  already, and whatever it picked up in layers 0, 1 and 2 is inside it. A head does
  not have to point at `trophy` to carry information about the trophy. That
  information can arrive in two short hops, `it` reading `because` and `because`
  having read `trophy`, with no single arrow from `it` to `trophy` anywhere in the
  picture.
- **You are looking at eight heads averaged together.** Averaging is the first thing
  anyone does and it hides disagreement. One head pointing hard at `trophy` while
  seven park on `</s>` averages away to nothing much. Section 5 takes the average
  apart, and this is the reason it exists.
- **Attention is only the reading step.** Each block is attention *and then* a
  feed-forward layer that works on each position separately — and nothing that
  feed-forward layer does appears in an attention picture at all. Most of this
  model's parameters are in those layers. Attention tells you where information was
  read from. It does not tell you what was done with it afterwards.

That last line was a claim on Tuesday. Here it is something you have watched.

**How to change the layer**

`LAYER = 3` in the cell above is the only thing you need to touch. Any value from
`0` to `5` works — there are six encoder layers, which the cell in section 1 printed
for you. Change it and re-run that one cell.

To look at a single head instead of the average of eight, change the line under it:

```python
LAYER, HEAD = 3, 0
enc_att = out.encoder_attentions[LAYER][0, HEAD].numpy()   # one head
# enc_att = out.encoder_attentions[LAYER][0].mean(axis=0).numpy()   # all eight, averaged
```

The indices are `[layer][batch, head]`, and the batch is always `0` because you are
sending one sentence at a time. Heads run `0` to `7`.

!!! tip "Your turn"

    Was it what you predicted? Most people predict *trophy* or *suitcase*.

    Work down the layers, `LAYER = 0` through `5`, and ignore `</s>` and the first
    token while you do it. Where does the leftover weight go? Early layers and late
    layers tend to behave differently — say how, in your own words.

    Then switch to single heads at the layer that looked most interesting. Is there
    a head doing something the average was hiding?

    Write a sentence of your own where a human needs the whole sentence to resolve a
    pronoun, and check whether anything in the encoder appears to do that work. **A
    confusion you can state precisely is a result.** "I looked at all forty-eight
    heads and none of them resolves the pronoun" is a finding, and it is most of the
    field's experience too.

---
## 4. Cross-attention, and whether it is an alignment

Cross-attention is the bridge: every German token can look at every English token, at
any distance, all at once. This is the thing an RNN could not do.

It is tempting to read it as a **word alignment** — a table of which German word
translates which English word. Some heads do look exactly like that. German is a good
test, because it moves the verb to the end of a subordinate clause: *"he bought a new
car yesterday"* becomes *"er gestern ein neues Auto **gekauft hat**"*. If
cross-attention is an alignment, the German verb at the end should reach back to
`bought` in the middle.

In [ ]:
english, german, out = look("I know that he bought a new car yesterday.")

LAYER, HEAD = 3, 4
cross = out.cross_attentions[LAYER][0, HEAD].numpy()  # [german, english]

heatmap(cross, german, english,
        f"Cross-attention — layer {LAYER}, head {HEAD}", figsize=(7, 6))

print(f"Where each German token looks (layer {LAYER}, head {HEAD}):\n")
for i, tok in enumerate(german):
    if tok == "<pad>":
        continue
    j = int(cross[i].argmax())
    print(f"  {clean([tok])[0]:<10} -> {clean([english[j]])[0]:<10} {cross[i, j]:.2f}")

!!! tip "Your turn"

    Find the two German tokens that both point back at `bought`. How far apart are
    they from it? That distance is the whole argument of the paper.

    Now change `LAYER` and `HEAD` and look again. **Layer 5, head 0 is worth your
    time**: it is systematically wrong in the same direction, and once you see the
    pattern you can say what it is doing instead.

    Most heads point most German tokens at `</s>`, the end-of-sentence token. If a
    head has nowhere useful to look, it has to put its weight somewhere, because each
    row must sum to 1. Does that change how much you trust an attention picture?

---
## 5. Heads do different things

Forty-eight encoder heads, six layers of eight. Averaging them was convenient and
wrong — they are not doing the same job.

You can score a head instead of squinting at it. Below is one crude score: how much
of a head's weight lands on *the token immediately before it*. A head that scores
near 1.0 is doing one legible thing and nothing else.

In [ ]:
PROBES = [
    "I know that he bought a new car yesterday.",
    "The trophy would not fit in the suitcase because it was too big.",
    "She said that the dog ate the whole cake.",
]


def head_scores(fn):
    """Average a per-head score over several sentences. fn(matrix) -> number."""
    totals = np.zeros((model.config.encoder_layers, model.config.encoder_attention_heads))
    for sentence in PROBES:
        _, _, o = look(sentence)
        for layer in range(totals.shape[0]):
            att = o.encoder_attentions[layer][0].numpy()
            for head in range(totals.shape[1]):
                totals[layer, head] += fn(att[head])
    return pd.DataFrame(
        totals / len(PROBES),
        index=[f"layer {i}" for i in range(totals.shape[0])],
        columns=[f"h{j}" for j in range(totals.shape[1])],
    ).round(2)


def previous_token(matrix):
    n = len(matrix)
    return float((matrix * np.eye(n, k=-1)).sum() / (n - 1))


def itself(matrix):
    return float(np.trace(matrix) / len(matrix))


print("How much each head attends to the PREVIOUS token:")
print(head_scores(previous_token))
print("\nHow much each head attends to ITSELF:")
print(head_scores(itself))

!!! tip "Your turn"

    One head in this model scores almost exactly 1.0 on *previous token* — it does
    nothing else, in any sentence. Find it, then draw it with `heatmap` and look at
    the line.

    Write your own score. `first_token`, `the_full_stop`, `attends_far_away` — a
    function that takes one matrix and returns one number is all it takes. Then find
    a head that scores high on it.

    **The harder half:** most heads will score low on everything you can think of.
    Pick one and try to say what it is doing. If you cannot, say that precisely —
    that is a real result, and it is most of the field's experience too.

---
## 6. Where the story breaks down

It is very easy to leave today believing that attention weights show you the model's
reasoning. They do not, and this model will prove it to you.

German pronouns carry gender, so a translation has to **commit** to what `it` refers
to: *die Trophäe* is feminine and takes **sie**, *der Koffer* is masculine and takes
**er**. The English hides the decision; the German cannot.

These two sentences have opposite correct answers. The trophy is too big — or the
suitcase is too small.

In [ ]:
PAIR = [
    "The trophy would not fit in the suitcase because it was too big.",
    "The trophy would not fit in the suitcase because it was too small.",
]

for sentence in PAIR:
    english, german, out = look(sentence)
    print(sentence)
    print("  ->", tokenizer.convert_tokens_to_string(german).replace("<pad>", "").strip())

    pronouns = [i for i, t in enumerate(german) if t in ("\u2581sie", "\u2581er")]
    if not pronouns:
        print("  no sie/er in this translation — try another sentence")
    for i in pronouns:
        cross = out.cross_attentions[3][0, 4].numpy()[i]
        top = cross.argsort()[::-1][:3]
        pairs = ", ".join(f"{clean([english[j]])[0]} {cross[j]:.2f}" for j in top)
        print(f"  the pronoun {clean([german[i]])[0]!r} attends to: {pairs}")
    print()

!!! tip "Your turn"

    Two things to notice, and they are not the same thing.

    **First:** does the German pronoun change between the two sentences? Should it?
    Work out what the correct German is for each before you decide whether the model
    is right.

    **Second:** whatever the model decided, look at where the attention went. Does
    any weight land on `trophy` or on `suitcase` at all? If the decision is not in
    the attention weights, where is it?

    This is the claim to leave with, and it is worth being able to argue: **an
    attention weight tells you where information was read from, not what was done
    with it.** Find your own example that makes the point, and bring it.

---
## 7. If you would rather click than plot

`bertviz` renders the same numbers as an interactive picture — every layer and head,
all three attention types, in one widget. Use the dropdowns; the tabs across the top
switch between encoder, decoder and cross.

It is a browser for the same tensors you have been slicing by hand, not a different
source of truth. If it fails to render, nothing above depends on it.

In [ ]:
from bertviz import model_view

english, german, out = look("The trophy would not fit in the suitcase because it was too big.")

model_view(
    encoder_attention=out.encoder_attentions,
    decoder_attention=out.decoder_attentions,
    cross_attention=out.cross_attentions,
    encoder_tokens=clean(english),
    decoder_tokens=clean(german),
)

---
## What to bring on Thursday

One finding. A claim plus the evidence you ran, in three sentences:

1. what you expected,
2. what the model actually did,
3. the layer, head and sentence, so somebody else can reproduce it.

A screenshot of a heatmap is fine. A table pasted into a document is fine. **A
confusion you can state precisely is a contribution** — "I found a head I cannot
explain, here it is" is a better ten minutes than a tidy result everybody nodded at.

Bring a laptop. If you have not opened this at all, come anyway and sit with someone
who has.